# 코드 설명

- **주제** : 악성코드 탐지 모델(MalConv)을 상대로 **적대적 예제**를 만들어, 딥러닝 탐지기의 강건성을 평가하기 (방어·연구 목적)
- **방법** : 파일 끝에 덧붙인 패딩 바이트만 FGSM(gradient ascent)으로 조금씩 바꿔, 탐지 점수를 "정상" 쪽으로 밀어붙임. 원본 실행 코드 영역은 건드리지 않음
- **사용 파일**
    - `MalGAN_input/malconv.h5` — 사전학습된 MalConv 탐지 모델 (저장소에 포함, 약 12MB)
    - `MalGAN_input/samplesIn.csv` — 입력 샘플 목록
    - 보조 모듈 : `MalGAN_preprocess.py`, `MalGAN_utils.py`, `MalGAN_gen_adv_examples.py`

> **⚠ 입력 샘플에 대한 중요 안내**
>
> 이 실습의 원본(교재)은 **악성코드 샘플 자체를 배포하지 않습니다.** 원본 `samplesIn.csv` 가 가리키던 악성 PE 파일들은 제공 자료에 들어 있지 않습니다(원본 목록은 `MalGAN_input/samplesIn_original.csv` 에 보존).
>
> 따라서 여기서는 **정상 PE 샘플 몇 개를 입력 대역(stand-in)으로** 사용해, 모델 로드 → 전처리 → 임베딩 → FGSM → 최근접 바이트 탐색 → 파일 저장까지 **파이프라인 전체가 실제로 동작하는지** 보여줍니다. 실제 악성코드 회피를 재현하려면 별도로 확보한 악성 샘플 경로를 `samplesIn.csv` 에 넣으면 됩니다.

> **최신화 메모 (TF1 → TF2 / Keras 3)**
>
> | 원본 (TF1, 동작 안 함) | 현재 (TF2) | 위치 |
> |---|---|---|
> | `keras.backend.tensorflow_backend.set_session` | 제거 | `MalGAN_utils.py` |
> | `tf.ConfigProto` / `tf.Session` | `tf.config` (GPU 메모리 growth) | `limit_gpu_memory` |
> | `K.gradients(loss, …)` | `tf.GradientTape` | `fgsm` |
> | `K.function([input], [emb])` | `keras.Model(input, emb)` 서브모델 | `build_emb_models` |
> | `keras.preprocessing.sequence.pad_sequences` | `keras.utils.pad_sequences` | `MalGAN_preprocess.py` |
>
> **malconv.h5 로드** : 이 `.h5` 는 구버전 Keras로 저장되어 최신 Keras 3에서는 바로 열리지 않습니다. `TF_USE_LEGACY_KERAS=1` 환경변수 + `tf-keras` 호환 패키지를 쓰면 구형 모델을 그대로 로드할 수 있습니다(이 환경변수는 **반드시 `tensorflow` 를 import 하기 전에** 설정해야 합니다).
>
> **MalConv 구조 주의** : 임베딩 뒤가 두 conv 분기를 곱하는 구조라, 레이어를 순서대로 다시 쌓는 방식으로는 서브모델을 만들 수 없습니다. 그래서 함수형 API로 중간 텐서(`emb_layer.output`)를 입력으로 지정해 `임베딩→출력` 서브모델을 잘라냈습니다.

In [1]:
# ── Google Colab 준비 셀 ─────────────────────────────────────────
# ★ malconv.h5 로드를 위해 tensorflow import 전에 환경변수를 먼저 설정합니다.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import subprocess
import sys

# PE 샘플(정상 파일, 약 440MB)을 올려둔 곳의 zip 직접 다운로드 주소.
# 비워 두면 다운로드를 건너뜁니다. (malconv.h5 는 저장소에 포함되어 있어 받을 필요 없음)
데이터_URL = "https://github.com/isl-hedgehog/python-cyber-seminar/releases/download/pe-samples-v1/pe_samples.zip"
데이터폴더 = ["Benign PE Samples", "Benign PE Samples UPX", "Benign PE Samples Amber"]

if "google.colab" in sys.modules:
    저장소 = "https://github.com/isl-hedgehog/python-cyber-seminar.git"
    이름 = 저장소.rsplit("/", 1)[-1].removesuffix(".git")
    실습폴더 = "02-adversarial-malware"
    if not os.path.isdir(f"/content/{이름}"):
        subprocess.run(["git", "clone", "-q", 저장소], cwd="/content", check=True)
    os.chdir(f"/content/{이름}/{실습폴더}")
    # Colab의 기본 TensorFlow는 Keras 3를 쓰므로, 구형 h5 로드를 위해 tf-keras를 설치합니다.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tf-keras"], check=True)
    print("Colab 준비 완료 :", os.getcwd())
else:
    print("로컬 환경 — 준비 셀을 건너뜁니다.")

# 입력 대역으로 쓸 정상 PE 샘플이 있는지 확인
if not os.path.isdir("Benign PE Samples"):
    print("⚠ 'Benign PE Samples' 폴더가 없습니다. 데이터_URL 을 설정하거나, 폴더를 이 위치에 두세요.")
else:
    print("입력 샘플 폴더 확인됨.")

로컬 환경 — 준비 셀을 건너뜁니다.
입력 샘플 폴더 확인됨.


### 1. 모듈 불러오기

`TF_USE_LEGACY_KERAS=1` 이 설정된 상태에서 `tensorflow.keras` 를 불러오면 구형 h5 를 읽을 수 있는 호환(tf-keras) 백엔드가 사용됩니다. 보조 모듈 3개도 함께 불러옵니다.

In [2]:
# 안전장치: 준비 셀을 건너뛰고 이 셀부터 실행하는 경우를 대비해 한 번 더 설정
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import pandas as pd
from tensorflow import keras

import MalGAN_utils
import MalGAN_gen_adv_examples

### 2. 설정값

- `pad_percent` : 파일 길이의 몇 %까지 패딩 바이트를 덧붙여 조작할지 (0.1 = 10%)
- `threshold` : 이 값보다 탐지 점수가 낮으면(=악성으로 판단하면) 공격을 시도 (0→악성, 1→정상)
- `step_size` : FGSM 한 걸음의 크기
- `limit` : GPU 메모리 제한 비율 (0 이면 제한하지 않음)

In [3]:
save_path = "MalGAN_output"
model_path = "MalGAN_input/malconv.h5"
log_path = "MalGAN_output/adversarial_log.csv"
pad_percent = 0.1
threshold = 0.6
step_size = 0.01
limit = 0.0
input_samples = "MalGAN_input/samplesIn.csv"

os.makedirs(save_path, exist_ok=True)
MalGAN_utils.limit_gpu_memory(limit)

### 3. 모델 로드와 적대적 예제 생성

입력 목록을 읽어 MalConv 모델을 로드한 뒤, 파일마다 적대적 예제를 생성합니다. 각 파일의 로그(원본 점수, 패딩 길이, 공격 후 점수)가 출력되고, 결과 바이트는 `MalGAN_output/` 에 저장됩니다.

- 원본 점수가 `threshold` 보다 **낮은**(악성으로 분류된) 파일만 공격 대상이 됩니다. 그 외에는 원본 바이트를 그대로 저장합니다.
- 정상 대역 샘플을 쓰므로, 점수가 높은 파일은 "공격 불필요"로 건너뛰는 동작을 함께 확인할 수 있습니다.

In [4]:
df = pd.read_csv(input_samples, header=None)
fn_list = df[0].values
model = keras.models.load_model(model_path)

adv_samples, log = MalGAN_gen_adv_examples.gen_adv_samples(
    model, fn_list, pad_percent, step_size, threshold
)

log.save(log_path)
for fn, adv in zip(fn_list, adv_samples):
    _fn = fn.split("/")[-1]
    dst = os.path.join(save_path, _fn)
    print(dst)
    with open(dst, "wb") as f:
        f.write(adv)


FILE: Benign PE Samples/c2wtshost.exe
	file length: 2626
	pad length: 262
	loss: 1.0
	score: 0.9999995827674866
	original score: 0.5894861221313477

FILE: Benign PE Samples/lpq.exe
	file length: 4708
	pad length: 470
	loss: nan
	score: nan
	original score: 0.9996594786643982

FILE: Benign PE Samples/dsamain.exe
	file length: 5048
	pad length: 504
	loss: nan
	score: nan
	original score: 0.9999767541885376

Log saved to "MalGAN_output/adversarial_log.csv"

MalGAN_output/c2wtshost.exe
MalGAN_output/lpq.exe
MalGAN_output/dsamain.exe


### 4. 결과 로그 확인

생성 과정에서 기록한 로그를 표로 확인합니다. `original score` 와 `predict score` 를 비교하면, 공격이 적용된 파일의 탐지 점수가 "정상" 쪽(값이 큼)으로 이동한 것을 볼 수 있습니다.

In [5]:
pd.read_csv(log_path)

,filename,original score,file length,pad length,loss,predict score
0,c2wtshost.exe,0.589486,2626,262,1.0,1.0
1,lpq.exe,0.999659,4708,470,NaN,NaN
2,dsamain.exe,0.999977,5048,504,NaN,NaN
